<a href="https://colab.research.google.com/github/Amsa02/skills-introduction-to-github/blob/main/Monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import urllib.parse
import requests
from bs4 import BeautifulSoup

# Configuration
TARGET_URL = "https://www.emys.app/youth/youth_train"
FLAG_FILE = "notified.txt"

# Keywords indicating the 4th delegation (Dec 25) is listed or open
KEYWORDS = [
    "الفوج الرابع",
    "25 ديسمبر",
    "٢٥ ديسمبر",
    "25/12",
    "٢٥/١٢",
    "25-12",
]

# Phrases indicating booking is closed/unavailable even if the date is mentioned
CLOSED_PHRASES = [
    "تم اكتمال العدد",
    "مغلق",
    "قريبا",
    "قريباً",
]


def send_whatsapp_alert(message: str) -> bool:
    phone = os.environ.get("WHATSAPP_PHONE")
    apikey = os.environ.get("CALLMEBOT_APIKEY")

    if not phone or not apikey:
        print("Error: WHATSAPP_PHONE or CALLMEBOT_APIKEY environment variables not set.")
        return False

    encoded_text = urllib.parse.quote(message)
    url = (
        f"https://api.callmebot.com/whatsapp.php"
        f"?phone={phone}&text={encoded_text}&apikey={apikey}"
    )

    try:
        response = requests.get(url, timeout=20)
        print(f"CallMeBot status: {response.status_code} | Response: {response.text}")
        return response.status_code == 200
    except requests.RequestException as exc:
        print(f"Failed to send WhatsApp message: {exc}")
        return False


def check_booking_status() -> None:
    if os.path.exists(FLAG_FILE):
        print("Alert already sent previously (notified.txt exists). Skipping check.")
        return

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/131.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "ar-EG,ar;q=0.9,en;q=0.8",
    }

    try:
        response = requests.get(TARGET_URL, headers=headers, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        print(f"Failed to fetch {TARGET_URL}: {exc}")
        sys.exit(0)  # Exit cleanly so GitHub Actions doesn't spam failure emails

    soup = BeautifulSoup(response.text, "html.parser")
    page_text = " ".join(soup.get_text(separator=" ").split())

    matched_keyword = next((kw for kw in KEYWORDS if kw in page_text), None)

    if not matched_keyword:
        print("4th delegation (Dec 25) not detected on the page yet.")
        return

    # Verify it isn't explicitly marked as full/closed right next to the match
    is_closed = any(closed in page_text for closed in CLOSED_PHRASES)
    if is_closed:
        print(f"Matched '{matched_keyword}', but closed/full status text is present.")
        return

    alert_msg = (
        "🚨 *Youth Train Alert!*\n"
        "Booking for the *4th Delegation (Dec 25)* to Luxor & Aswan appears to be LIVE!\n"
        f"Book immediately: {TARGET_URL}"
    )

    if send_whatsapp_alert(alert_msg):
        with open(FLAG_FILE, "w", encoding="utf-8") as f:
            f.write("notified\n")
        print("Notification sent and flag file created.")


if __name__ == "__main__":
    check_booking_status()